# 差分メモ

- 対象: `[2-8]dpc-starter-train-v2-colab.ipynb`
- 元: `[2-9]dpc-starter-train-v2-colab.ipynb`
- 種別: モデル差し替え（ByT5-small）
- 変更点:
  - `MODEL_NAME` を `google/byt5-small` に変更
  - 推定/本学習の出力ディレクトリ名を `byt5-small` に変更
- 変更理由/仮説:
  - ByT5-small で同じ 2 段階学習（estimate→full）を回して、学習量見積もりの運用を維持する


# Deep Past Initiative – Machine Translation (Training Notebook)

This notebook is a **starter / baseline** for this Kaggle competition.

Main ideas:
- Use **ByT5** to handle noisy Akkadian transliterations at the character level
- Perform **simple sentence alignment** to increase training data
- Fine-tune using HuggingFace `Trainer`


Inference Code is [here](https://www.kaggle.com/code/takamichitoda/dpc-starter-infer).

In [ ]:
# Colab セットアップ（再現性のためバージョン固定）
!pip -q uninstall -y transformers
!pip -q install transformers==4.57.1 datasets accelerate evaluate sacrebleu openpyxl


In [ ]:
from google.colab import drive
drive.mount("/content/gdrive")


In [ ]:
import torch
import transformers
import datasets

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import gc
import re
import shutil
import math
import unicodedata
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback,
)

In [ ]:
import os
import re
from datetime import datetime

# Google Drive 上の作業ディレクトリ（必要に応じて変更）
DRIVE_ROOT = "/content/gdrive/MyDrive/Kaggle/Deep_Past_Challenge"
OUTPUT_ROOT = f"{DRIVE_ROOT}/models"

def _sanitize(name: str) -> str:
    name = re.sub(r"[^A-Za-z0-9._-]+", "_", str(name))
    name = name.strip("_")
    return name or "run"

def get_colab_notebook_name(default: str) -> str:
    """Colab上で開いている .ipynb の名前を取れれば取る（取れなければ default）。"""
    try:
        import ipykernel
        import requests
        import json
        import os

        connection_file = os.path.basename(ipykernel.get_connection_file())
        kernel_id = connection_file.split("-", 1)[1].split(".")[0]
        sessions = requests.get("http://172.28.0.12:9000/api/sessions")
        sessions = json.loads(sessions.text)
        for session in sessions:
            if session.get("kernel", {}).get("id") == kernel_id:
                path = session.get("path", "")
                name = os.path.splitext(os.path.basename(path))[0]
                if name:
                    return name
    except Exception:
        pass
    return default

_NOTEBOOK_NAME = _sanitize(get_colab_notebook_name(default="[2-8]dpc-starter-train-v2-colab"))
_TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

class Config:
    # ByT5-small を学習
    MODEL_NAME = "google/byt5-small"

    # 長文分割済みデータを活かすため、学習・推論とも 1024 に揃える
    MAX_LENGTH = 1024

    # Training
    # まずは Colab で回しやすい設定（必要なら上げる）
    EPOCHS = 10
    LEARNING_RATE = 2e-4

    # メモリに応じて grad-accum で effective batch を調整
    PER_DEVICE_TRAIN_BATCH_SIZE = 16
    GRADIENT_ACCUMULATION_STEPS = 1

    # --- Early-stopping 用の検証 split ---
    # leakage を避けるため、`oare_id` グループ単位で hold-out します
    VAL_GROUP_FRACTION = 0.05
    VAL_SEED = 42
    VAL_ONLY_AKK2EN = True

    # Early-stopping
    MAX_EPOCHS_ESTIMATE = 30  # 上限（ここまで回す前に止まる想定）
    EARLY_STOPPING_PATIENCE = 3
    EARLY_STOPPING_THRESHOLD = 0.0
    EVAL_STEPS = 200
    PER_DEVICE_EVAL_BATCH_SIZE = 16

    # Output（Driveに永続化）
    # 出力（Driveに永続化）
    # - 推定(early-stopping) と full-train の出力を分ける
    RUN_DIR = os.path.join(OUTPUT_ROOT, f"{_NOTEBOOK_NAME}_{_TIMESTAMP}")
    OUTPUT_DIR_ESTIMATE = os.path.join(RUN_DIR, "estimate_es_byt5-small_multitask")
    OUTPUT_DIR_FULL = os.path.join(RUN_DIR, "fulltrain_es_byt5-small_multitask")


In [ ]:
# Fix the seed (for reproducibility).
def seed_everything(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    torch.cuda.manual_seed(seed)

seed_everything()

In [ ]:
import os
import pandas as pd

# 公式データを Drive に置いた前提
# 例: /content/gdrive/MyDrive/Kaggle/Deep_Past_Challenge/data/kaggle/deep-past-initiative-machine-translation/train.csv
DRIVE_ROOT = "/content/gdrive/MyDrive/Kaggle/Deep_Past_Challenge"
INPUT_DIR = f"{DRIVE_ROOT}/data/kaggle/deep-past-initiative-machine-translation"

# ------------------------------------------
# Train: curated xlsx を優先して読む
# ------------------------------------------
# curated xlsx も Drive に置いて参照してください。
USE_CURATED_TRAIN = True
CURATED_TRAIN_XLSX_CANDIDATES = [
    "/content/gdrive/MyDrive/Kaggle/Deep_Past_Challenge/data/PROCESS_DATA/train.curated.v002-6.xlsx",
]


def load_train_df():
    if USE_CURATED_TRAIN:
        curated_path = None
        for p in CURATED_TRAIN_XLSX_CANDIDATES:
            if os.path.exists(p):
                curated_path = p
                break
        if curated_path is None:
            raise FileNotFoundError(
                "curated train xlsx が見つかりません。CURATED_TRAIN_XLSX_CANDIDATES を確認してください。"
            )

        df = pd.read_excel(curated_path)

        unnamed_cols = [c for c in df.columns if str(c).startswith("Unnamed")]
        if unnamed_cols:
            df = df.drop(columns=unnamed_cols)

        df = df.rename(columns={"transliteration_x": "transliteration"})

        required_cols = {"oare_id", "transliteration", "translation"}
        missing_cols = required_cols - set(df.columns)
        assert not missing_cols, f"curated xlsx の必須カラムが不足しています: {sorted(missing_cols)}"

        df = df[["oare_id", "transliteration", "translation"]].copy()
        df["transliteration"] = df["transliteration"].fillna("").astype(str)
        df["translation"] = df["translation"].fillna("").astype(str)

        df.attrs["train_source"] = curated_path
        return df

    df = pd.read_csv(f"{INPUT_DIR}/train.csv")
    df.attrs["train_source"] = f"{INPUT_DIR}/train.csv"
    return df


train_df = load_train_df()
test_df = pd.read_csv("/content/gdrive/MyDrive/Kaggle/Deep_Past_Challenge/data/test.csv")


In [ ]:
train_source = train_df.attrs.get("train_source", "(unknown)")
print(f"Train Data: {len(train_df)} docs | source={train_source}")

In [ ]:
def simple_sentence_aligner(df):
    """
    【戦略の肝】
    Trainデータの「文書(複数文)」を、Testデータと同じ「文(1文)」に分割します。
    ここでは「英語の文数」と「アッカド語の行数」が一致する場合のみ分割する
    というヒューリスティック（簡易ルール）を使います。

    NOTE:
      - Sentence splitting increases samples per original document.
      - To avoid leakage in CV, we keep `oare_id` as a grouping key.
    """
    aligned_data = []

    for _, row in df.iterrows():
        oare_id = row["oare_id"]
        src = str(row["transliteration"])
        tgt = str(row["translation"])

        # Split the English text by sentence-ending punctuation.
        tgt_sents = [t.strip() for t in re.split(r"(?<=[.!?])\s+", tgt) if t.strip()]

        # Assume the Akkadian text is often separated by newlines and split accordingly.
        src_lines = [s.strip() for s in src.split('\n') if s.strip()]

        # If the counts match, trust it as 1-to-1 pairs and use the split version.
        if len(tgt_sents) > 1 and len(tgt_sents) == len(src_lines):
            for s, t in zip(src_lines, tgt_sents):
                if len(s) > 3 and len(t) > 3:  # Remove junk/noisy data.
                    aligned_data.append({"oare_id": oare_id, "transliteration": s, "translation": t})
        else:
            # If splitting fails (counts don't match), keep the original document pair as-is (safe fallback).
            aligned_data.append({"oare_id": oare_id, "transliteration": src, "translation": tgt})

    return pd.DataFrame(aligned_data, columns=["oare_id", "transliteration", "translation"])


In [ ]:
# Run data augmentation.
train_expanded = simple_sentence_aligner(train_df)
print(f"Expanded Train Data: {len(train_expanded)} sentences (Alignment applied)")

train_expanded = train_expanded.reset_index(drop=True)
assert set(["oare_id", "transliteration", "translation"]).issubset(train_expanded.columns)

# ------------------------------------------
# Prefix multi-task: akk→en + en→akk (2x)
# ------------------------------------------
multitask_df = pd.concat(
    [
        train_expanded.assign(
            task="akk2en",
            source=train_expanded["transliteration"],
            target=train_expanded["translation"],
        ),
        train_expanded.assign(
            task="en2akk",
            source=train_expanded["translation"],
            target=train_expanded["transliteration"],
        ),
    ],
    ignore_index=True,
)[["oare_id", "task", "source", "target"]]

print(f"Multi-task Train Data: {len(multitask_df)} samples (2x directions)")
# Deterministic shuffle so batches mix directions
multitask_df = multitask_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(multitask_df.head())


In [ ]:
# ==========================================
# 3. Tokenization & preprocessing
# ==========================================
from typing import List

tokenizer = AutoTokenizer.from_pretrained(Config.MODEL_NAME)

PREFIX_AKK_EN = "translate Akkadian to English: "
PREFIX_EN_AKK = "translate English to Akkadian: "

# --- Pre/Post processing (aligned to notebooks/006/lb-35-9-ensembling-post-processing-baseline.ipynb) ---
_V2 = re.compile(r"([aAeEiIuU])(?:2|₂)")
_V3 = re.compile(r"([aAeEiIuU])(?:3|₃)")
_ACUTE = str.maketrans({"a":"á","e":"é","i":"í","u":"ú","A":"Á","E":"É","I":"Í","U":"Ú"})
_GRAVE = str.maketrans({"a":"à","e":"è","i":"ì","u":"ù","A":"À","E":"È","I":"Ì","U":"Ù"})

def _ascii_to_diacritics(s: str) -> str:
    s = s.replace("sz", "š").replace("SZ", "Š")
    s = s.replace("s,", "ṣ").replace("S,", "Ṣ")
    s = s.replace("t,", "ṭ").replace("T,", "Ṭ")
    s = _V2.sub(lambda m: m.group(1).translate(_ACUTE), s)
    s = _V3.sub(lambda m: m.group(1).translate(_GRAVE), s)
    return s


# Unicode fraction map (all single-character Unicode vulgar fractions)
_UNICODE_FRAC_MAP = {
    (1, 2): "½", (1, 3): "⅓", (2, 3): "⅔",
    (1, 4): "¼", (3, 4): "¾",
    (1, 5): "⅕", (2, 5): "⅖", (3, 5): "⅗", (4, 5): "⅘",
    (1, 6): "⅙", (5, 6): "⅚",
    (1, 7): "⅐",
    (1, 8): "⅛", (3, 8): "⅜", (5, 8): "⅝", (7, 8): "⅞",
    (1, 9): "⅑",
    (1, 10): "⅒",
}
def _trunc_float(x: float, places: int = 4) -> float:
    factor = 10 ** places
    if x >= 0:
        return math.floor(x * factor) / factor
    return -math.floor(-x * factor) / factor

def _fmt_trunc(x: float, places: int = 4) -> str:
    return f"{_trunc_float(x, places):.{places}f}".rstrip("0").rstrip(".")

# Map the 4-decimal *truncated* representation to a Unicode fraction.
# (Host example: 1.333330000... -> 1.3333; 0.1666 -> ⅙)
_FRAC_DECIMAL_MAP_4 = {}
for (n, d), ch in _UNICODE_FRAC_MAP.items():
    _FRAC_DECIMAL_MAP_4[_fmt_trunc(n / d, 4)] = ch

# Decimals (incl. long floats)
_DECIMAL_RE = re.compile(r"(?<![\w/])(\d+\.\d+)(?![\w/])")
_LONG_DECIMAL_RE = re.compile(r"(?<![\w/])(\d+\.\d{5,})(?![\w/])")

# General fractions like "1/2" or mixed fractions like "2 1/2".
_MIXED_FRAC_RE = re.compile(r"(?<![\w/])(\d+)\s+(\d+)\s*/\s*(\d+)(?![\w/])")
_SIMPLE_FRAC_RE = re.compile(r"(?<![\w/])(\d+)\s*/\s*(\d+)(?![\w/])")

def _mixed_frac_repl(m: re.Match) -> str:
    ip = int(m.group(1))
    num = int(m.group(2))
    den = int(m.group(3))
    ch = _UNICODE_FRAC_MAP.get((num, den))
    if ch:
        return f"{ip} {ch}" if ip != 0 else ch
    return m.group(0)

def _simple_frac_repl(m: re.Match) -> str:
    num = int(m.group(1))
    den = int(m.group(2))
    ch = _UNICODE_FRAC_MAP.get((num, den))
    if ch:
        return ch
    return m.group(0)

def _canon_decimal_str(s: str) -> str:
    # Keep x.5 (e.g., 2.5) as-is by request.
    if re.fullmatch(r"[1-9]\d*\.5", s):
        return s

    m = re.fullmatch(r"(\d+)\.(\d+)", s)
    if not m:
        return s

    ip = int(m.group(1))
    frac_digits = m.group(2)

    # Truncate fractional part to 4 digits (no rounding), pad with zeros for lookup.
    frac4 = (frac_digits + "0000")[:4]
    frac_key = ("0." + frac4).rstrip("0").rstrip(".")
    ch = _FRAC_DECIMAL_MAP_4.get(frac_key)
    if ch and frac_key != "0":
        if ip == 0:
            return ch
        return f"{ip} {ch}"

    # Long-float shortening: truncate to 4 digits after the decimal.
    if len(frac_digits) > 4:
        return f"{ip}.{frac4}".rstrip("0").rstrip(".")

    return s


_TAG_BIGGAP_RE = re.compile(r"<\s*big[\s_\-]*gap\s*>", re.I)
_TAG_GAP_RE    = re.compile(r"<\s*gap\s*>", re.I)
_BARE_BIGGAP   = re.compile(r"\bbig[\s_\-]*gap\b", re.I)
_ELLIPSIS_RE   = re.compile(r"(?:\.{3,}|…+|\[\.+\])")
_BRACKET_X_RE  = re.compile(r"(\[\s*x\s*\]|\(\s*x\s*\))", re.I)
_XTOKEN_RUN_RE = re.compile(r"\bx(?:\s+x)+\b", re.I)
_XRUN_RE       = re.compile(r"(?<!\w)x{2,}(?!\w)", re.I)
_XTOK_RE       = re.compile(r"(?<!\w)x(?!\w)", re.I)
_WS_RE         = re.compile(r"\s+")

_GAP_UNIFIED_RE = re.compile(
    r"<\s*big[\s_\-]*gap\s*>"
    r"|<\s*gap\s*>"
    r"|\bbig[\s_\-]*gap\b"
    r"|\bx(?:\s+x)+\b"
    r"|\.{3,}|…+|\[\.+\]"
    r"|\[\s*x\s*\]|\(\s*x\s*\)"
    r"|(?<!\w)x{2,}(?!\w)"
    r"|(?<!\w)x(?!\w)"
    r"|\(\s*large\s+break\s*\)"
    r"|\(\s*break\s*\)"
    r"|\(\s*\d+\s+broken\s+lines?\s*\)",
    re.I
)

def _normalize_gaps_vec(ser: pd.Series) -> pd.Series:
    return ser.str.replace(_GAP_UNIFIED_RE, "<gap>", regex=True)


_CHAR_TRANS = str.maketrans({
    "ḫ":"h","Ḫ":"H","ʾ":"",
    "₀":"0","₁":"1","₂":"2","₃":"3","₄":"4",
    "₅":"5","₆":"6","₇":"7","₈":"8","₉":"9",
    "—":"-","–":"-",
})
_SUB_X = "ₓ"

_UNICODE_UPPER = r"A-ZŠṬṢḪ\u00C0-\u00D6\u00D8-\u00DE\u0160\u1E00-\u1EFF"
_UNICODE_LOWER = r"a-zšṭṣḫ\u00E0-\u00F6\u00F8-\u00FF\u0161\u1E01-\u1EFF"

_DET_UPPER_RE = re.compile(r"\(([" + _UNICODE_UPPER + r"0-9]{1,6})\)")
_DET_LOWER_RE = re.compile(r"\(([" + _UNICODE_LOWER + r"]{1,4})\)")

_PN_RE = re.compile(r"\bPN\b")
_KUBABBAR_RE = re.compile(r"KÙ\.B\.")


class OptimizedPreprocessor:
    def preprocess_batch(self, texts: List[str]) -> List[str]:
        ser = pd.Series(texts).fillna("").astype(str)
        ser = ser.apply(_ascii_to_diacritics)

        # Uppercase determinatives are unwrapped, lowercase ones are converted to braces.
        ser = ser.str.replace(_DET_UPPER_RE, r"\1", regex=True)
        ser = ser.str.replace(_DET_LOWER_RE, r"{\1}", regex=True)

        ser = _normalize_gaps_vec(ser)
        ser = ser.str.translate(_CHAR_TRANS)
        ser = ser.str.replace(_SUB_X, "", regex=False)
        ser = ser.str.replace(_KUBABBAR_RE, "KÙ.BABBAR", regex=True)
        ser = ser.str.replace(_MIXED_FRAC_RE, _mixed_frac_repl, regex=True)
        ser = ser.str.replace(_SIMPLE_FRAC_RE, _simple_frac_repl, regex=True)
        ser = ser.str.replace(_DECIMAL_RE, lambda m: _canon_decimal_str(m.group(1)), regex=True)
        ser = ser.str.replace(_WS_RE, " ", regex=True).str.strip()
        return ser.tolist()

_SOFT_GRAM_RE = re.compile(
    r"\(\s*(?:fem|plur|pl|sing|singular|plural|\?|\!)"
    r"(?:\.\s*(?:plur|plural|sing|singular))?"
    r"\.?\s*[^)]*\)", re.I
)
_BARE_GRAM_RE = re.compile(r"(?<!\w)(?:fem|sing|pl|plural)\.?(?!\w)\s*", re.I)
_UNCERTAIN_RE = re.compile(r"\(\?\)")
_CURLY_QUOTES_RE = re.compile("[\u201c\u201d\u2018\u2019]")

_MONTH_RE = re.compile(r"\bMonth\s+(XII|XI|X|IX|VIII|VII|VI|V|IV|III|II|I)\b", re.I)
_ROMAN2INT = {"I":1,"II":2,"III":3,"IV":4,"V":5,"VI":6,"VII":7,"VIII":8,"IX":9,"X":10,"XI":11,"XII":12}

_REPEAT_WORD_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+")
_REPEAT_PUNCT_RE = re.compile(r"([.,])\1+")
_PUNCT_SPACE_RE = re.compile(r"\s+([.,:])")

_FORBIDDEN_TRANS = str.maketrans("", "", '()——<>⌈⌋⌊[]+ʾ;')

_COMMODITY_RE = re.compile(r'-(gold|tax|textiles)\b')
_COMMODITY_REPL = {
    "gold": "pašallum gold",
    "tax": "šadduātum tax",
    "textiles": "kutānum textiles",
}

def _commodity_repl(m: re.Match) -> str:
    return _COMMODITY_REPL[m.group(1)]


_SHEKEL_REPLS = [
    (re.compile(r'5\s+11\s*/\s*12\s+shekels?', re.I), '6 shekels less 15 grains'),
    (re.compile(r'5\s*/\s*12\s+shekels?', re.I), '⅔ shekel 15 grains'),
    (re.compile(r'7\s*/\s*12\s+shekels?', re.I), '½ shekel 15 grains'),
    (re.compile(r'1\s*/\s*12\s*(?:\(shekel\)|\bshekel)?', re.I), '15 grains'),
]

_ALT_SLASH_PAIR_RE = re.compile(r"\b([^\W\d_]+)\s*/\s*([^\W\d_]+)\b")
_STRAY_MARKS_RE = re.compile(r'<<[^>]*>>|<(?!gap\b)[^>]*>')
_MULTI_GAP_RE = re.compile(r'(?:<gap>\s*){2,}')

def _month_repl(m: re.Match) -> str:
    return f"Month {_ROMAN2INT.get(m.group(1).upper(), m.group(1))}"


class VectorizedPostprocessor:
    def postprocess_batch(self, translations: List[str]) -> List[str]:
        s = pd.Series(translations).fillna("").astype(str)

        s = _normalize_gaps_vec(s)
        s = s.str.replace(_PN_RE, "<gap>", regex=True)
        s = s.str.replace(_COMMODITY_RE, _commodity_repl, regex=True)

        for pat, repl in _SHEKEL_REPLS:
            s = s.str.replace(pat, repl, regex=True)

        s = s.str.replace(_MIXED_FRAC_RE, _mixed_frac_repl, regex=True)
        s = s.str.replace(_SIMPLE_FRAC_RE, _simple_frac_repl, regex=True)
        s = s.str.replace(_DECIMAL_RE, lambda m: _canon_decimal_str(m.group(1)), regex=True)

        s = s.str.replace(_SOFT_GRAM_RE, " ", regex=True)
        s = s.str.replace(_BARE_GRAM_RE, " ", regex=True)
        s = s.str.replace(_UNCERTAIN_RE, "", regex=True)

        s = s.str.replace(_STRAY_MARKS_RE, "", regex=True)
        # Keep the left alternative (e.g., "you / she" -> "you")
        s = s.str.replace(_ALT_SLASH_PAIR_RE, r"\1", regex=True)

        # Remove only curly quotes; keep straight quotes and apostrophes.
        s = s.str.replace(_CURLY_QUOTES_RE, "", regex=True)

        s = s.str.replace(_MONTH_RE, _month_repl, regex=True)
        s = s.str.replace(_MULTI_GAP_RE, "<gap>", regex=True)

        s = s.str.replace("<gap>", "\x00GAP\x00", regex=False)
        s = s.str.translate(_FORBIDDEN_TRANS)
        s = s.str.replace("\x00GAP\x00", " <gap> ", regex=False)

        s = s.str.replace(_REPEAT_WORD_RE, r"\1", regex=True)
        for n in range(4, 1, -1):
            pat = r"\b((?:\w+\s+){" + str(n-1) + r"}\w+)(?:\s+\1\b)+"
            s = s.str.replace(pat, r"\1", regex=True)

        s = s.str.replace(_PUNCT_SPACE_RE, r"\1", regex=True)
        s = s.str.replace(_REPEAT_PUNCT_RE, r"\1", regex=True)
        s = s.str.replace(_WS_RE, " ", regex=True).str.strip()

        return s.tolist()

_preprocessor = OptimizedPreprocessor()
_postprocessor = VectorizedPostprocessor()


def preprocess_function(examples):
    tasks = examples["task"]
    sources = examples["source"]
    targets = examples["target"]

    inputs = []
    out_targets = []

    # Batch-precompute by task to keep vectorized pandas ops fast.
    idx_akk2en = [i for i, t in enumerate(tasks) if t == "akk2en"]
    idx_en2akk = [i for i, t in enumerate(tasks) if t == "en2akk"]

    src_akk = [sources[i] for i in idx_akk2en]
    tgt_en = [targets[i] for i in idx_akk2en]

    src_en = [sources[i] for i in idx_en2akk]
    tgt_akk = [targets[i] for i in idx_en2akk]

    src_akk_pp = _preprocessor.preprocess_batch(src_akk) if src_akk else []
    tgt_en_pp = _postprocessor.postprocess_batch(tgt_en) if tgt_en else []

    # English input is also normalized using the same postprocessor for consistency.
    src_en_pp = _postprocessor.postprocess_batch(src_en) if src_en else []
    tgt_akk_pp = _preprocessor.preprocess_batch(tgt_akk) if tgt_akk else []

    it_src_akk = iter(src_akk_pp)
    it_tgt_en = iter(tgt_en_pp)
    it_src_en = iter(src_en_pp)
    it_tgt_akk = iter(tgt_akk_pp)

    for t in tasks:
        if t == "akk2en":
            s_norm = next(it_src_akk)
            y_norm = next(it_tgt_en)
            inputs.append(PREFIX_AKK_EN + s_norm)
            out_targets.append(y_norm)
        elif t == "en2akk":
            s_norm = next(it_src_en)
            y_norm = next(it_tgt_akk)
            inputs.append(PREFIX_EN_AKK + s_norm)
            out_targets.append(y_norm)
        else:
            raise ValueError(f"Unknown task: {t}")

    model_inputs = tokenizer(inputs, max_length=Config.MAX_LENGTH, truncation=True)
    labels = tokenizer(out_targets, max_length=Config.MAX_LENGTH, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


In [ ]:
# ==========================================
# 4. 検証 split で学習量を見積もる（EarlyStopping）
# ==========================================
import math
import numpy as np

# ---- hold-out split（oare_id グループ単位）----
groups = train_expanded["oare_id"].dropna().astype(str).unique()
assert len(groups) > 0
rng = np.random.RandomState(Config.VAL_SEED)
val_group_size = max(1, int(round(len(groups) * Config.VAL_GROUP_FRACTION)))
val_groups = set(rng.choice(groups, size=val_group_size, replace=False))

df_est_train = multitask_df[~multitask_df["oare_id"].astype(str).isin(val_groups)].copy()
df_est_val = multitask_df[multitask_df["oare_id"].astype(str).isin(val_groups)].copy()
if Config.VAL_ONLY_AKK2EN:
    df_est_val = df_est_val[df_est_val["task"] == "akk2en"].copy()

df_est_train = df_est_train.reset_index(drop=True)
df_est_val = df_est_val.reset_index(drop=True)
print(f"[estimate] groups: total={len(groups)} val={len(val_groups)}")
print(f"[estimate] train={len(df_est_train)} val={len(df_est_val)} (VAL_ONLY_AKK2EN={Config.VAL_ONLY_AKK2EN})")

# ---- Dataset + tokenize ----
ds_est_train = Dataset.from_pandas(df_est_train)
ds_est_val = Dataset.from_pandas(df_est_val)

tok_est_train = ds_est_train.map(
    preprocess_function,
    batched=True,
    remove_columns=ds_est_train.column_names,
)
tok_est_val = ds_est_val.map(
    preprocess_function,
    batched=True,
    remove_columns=ds_est_val.column_names,
)

# ---- Model ----
model_est = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)
model_est.config.use_cache = False
model_est.generation_config.max_length = Config.MAX_LENGTH
model_est.gradient_checkpointing_enable()
data_collator_est = DataCollatorForSeq2Seq(tokenizer, model=model_est)

os.makedirs(Config.OUTPUT_DIR_ESTIMATE, exist_ok=True)

# --- Colab mixed precision / TF32（A100想定だが自動で切替） ---
use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16
use_tf32 = torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8
if use_tf32:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print(f"precision: bf16={use_bf16} fp16={use_fp16} tf32={use_tf32}")

# ---- steps/epoch の概算（single GPU 前提）----
effective_batch = Config.PER_DEVICE_TRAIN_BATCH_SIZE * Config.GRADIENT_ACCUMULATION_STEPS
steps_per_epoch_est = int(math.ceil(len(tok_est_train) / effective_batch))
print(f"[estimate] effective_batch={effective_batch} steps/epoch≈{steps_per_epoch_est}")

args_est = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR_ESTIMATE,
    eval_strategy="steps",
    eval_steps=Config.EVAL_STEPS,
    save_strategy="steps",
    save_steps=Config.EVAL_STEPS,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    learning_rate=Config.LEARNING_RATE,
    bf16=use_bf16,
    fp16=use_fp16,
    tf32=use_tf32,
    gradient_checkpointing=True,
    group_by_length=True,
    per_device_train_batch_size=Config.PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=Config.PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,
    weight_decay=0.01,
    num_train_epochs=Config.MAX_EPOCHS_ESTIMATE,
    predict_with_generate=False,
    logging_strategy="steps",
    logging_steps=200,
    disable_tqdm=True,
    report_to="none",
)

trainer_est = Seq2SeqTrainer(
    model=model_est,
    args=args_est,
    train_dataset=tok_est_train,
    eval_dataset=tok_est_val,
    data_collator=data_collator_est,
    tokenizer=tokenizer,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=Config.EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=Config.EARLY_STOPPING_THRESHOLD,
        )
    ],
)

print("Starting Training (estimate w/ early stopping, monitor=eval_loss)...")
trainer_est.train()

best_ckpt = trainer_est.state.best_model_checkpoint
best_step = None
if best_ckpt and "checkpoint-" in best_ckpt:
    try:
        best_step = int(best_ckpt.rsplit("checkpoint-", 1)[-1])
    except Exception:
        best_step = None
if best_step is None:
    best_step = int(trainer_est.state.global_step)

best_epoch_equiv = best_step / max(1, steps_per_epoch_est)
print(f"[estimate] best_ckpt={best_ckpt}")
print(f"[estimate] best_step={best_step} best_epoch_equiv≈{best_epoch_equiv:.3f} (using steps/epoch≈{steps_per_epoch_est})")

# best model はすでにメモリへ再読込されているので、checkpoint は Drive に残さない
for name in os.listdir(Config.OUTPUT_DIR_ESTIMATE):
    path = os.path.join(Config.OUTPUT_DIR_ESTIMATE, name)
    if name.startswith("checkpoint-") and os.path.isdir(path):
        shutil.rmtree(path)
print(f"[estimate] removed checkpoint dirs under: {Config.OUTPUT_DIR_ESTIMATE}")

# estimate フェーズの GPU/CPU メモリを明示的に解放してから full train へ進む
model_est.cpu()
del trainer_est, model_est, data_collator_est
del tok_est_train, tok_est_val, ds_est_train, ds_est_val
del df_est_train, df_est_val
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print("[estimate] released estimate-phase memory")

# ==========================================
# 5. 全 train で best_epoch_equiv 相当だけ学習して保存
# ==========================================
ds_full = Dataset.from_pandas(multitask_df)
tok_full = ds_full.map(
    preprocess_function,
    batched=True,
    remove_columns=ds_full.column_names,
)

steps_per_epoch_full = int(math.ceil(len(tok_full) / effective_batch))
full_max_steps = int(max(1, round(best_epoch_equiv * steps_per_epoch_full)))
print(f"[full] samples={len(tok_full)} steps/epoch≈{steps_per_epoch_full} -> max_steps={full_max_steps}")

model_full = AutoModelForSeq2SeqLM.from_pretrained(Config.MODEL_NAME)
model_full.config.use_cache = False
model_full.generation_config.max_length = Config.MAX_LENGTH
model_full.gradient_checkpointing_enable()
data_collator_full = DataCollatorForSeq2Seq(tokenizer, model=model_full)

os.makedirs(Config.OUTPUT_DIR_FULL, exist_ok=True)

args_full = Seq2SeqTrainingArguments(
    output_dir=Config.OUTPUT_DIR_FULL,
    eval_strategy="no",
    save_strategy="no",
    learning_rate=Config.LEARNING_RATE,
    bf16=use_bf16,
    fp16=use_fp16,
    tf32=use_tf32,
    gradient_checkpointing=True,
    group_by_length=True,
    per_device_train_batch_size=Config.PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=Config.GRADIENT_ACCUMULATION_STEPS,
    weight_decay=0.01,
    max_steps=full_max_steps,
    num_train_epochs=Config.EPOCHS,
    predict_with_generate=False,
    logging_strategy="steps",
    logging_steps=200,
    disable_tqdm=True,
    report_to="none",
)

trainer_full = Seq2SeqTrainer(
    model=model_full,
    args=args_full,
    train_dataset=tok_full,
    data_collator=data_collator_full,
    tokenizer=tokenizer,
)

print("Starting Training (full train, steps estimated from early stopping)...")
trainer_full.train()

trainer_full.save_model(Config.OUTPUT_DIR_FULL)
tokenizer.save_pretrained(Config.OUTPUT_DIR_FULL)
print(f"Saved final model to: {Config.OUTPUT_DIR_FULL}")


In [ ]:
# --- Notes ---
# このノートブックは 2 段階で ByT5-small を学習します。
#  1) train の一部を検証（oare_id hold-out）として `eval_loss` を監視し、EarlyStopping で最適学習量を見積もる
#  2) 1) の best_epoch 相当を full train の step 数に換算し、その step 数だけ全データで学習して最終モデルを保存する
#
# Input（Drive想定）:
#   - 公式データ: DRIVE_ROOT/data/kaggle/deep-past-initiative-machine-translation/
#   - curated xlsx: CURATED_TRAIN_XLSX_CANDIDATES のどれかに配置
#
# Output（Drive保存）:
#   - 推定用: Config.OUTPUT_DIR_ESTIMATE（学習中のみ checkpoint を作り、best_step 算出後に削除）
#   - estimate フェーズ完了後に `trainer_est` / `model_est` / tokenized dataset を解放してから full train へ進む
#   - 最終:   Config.OUTPUT_DIR_FULL（save_model/tokenizer）
